In [39]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [53]:
np.random.seed(42)

# 1. GENERATE RANDOM CLASSIFICATION DATA
# Create a dataset with 1000 samples, 10 features, and 2 classes (binary)
X_raw, y_raw = make_classification(
    n_samples=100, n_features=10, n_classes=2, random_state=42
)

# Split into 80% train and 20% test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Scale features for better neural network convergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Now convert these sets to tensors, ensuring float64 for consistency
X_train = torch.from_numpy(X_train).to(torch.float64)
X_test = torch.from_numpy(X_test).to(torch.float64)

y_train = torch.from_numpy(y_train).to(torch.float64)
y_test = torch.from_numpy(y_test).to(torch.float64)

In [55]:
# Now we will create our simpleNN model
# We will have a simple NN where there are 10 inputs to a single neuron and with a single bias and a 2 classification problem.
class SimpleNN:
  def __init__(self,X,y):
    self.weights = torch.randn(X.shape[1],1,requires_grad=True,dtype=torch.float64)
    self.bias = torch.zeros(1,requires_grad=True,dtype=torch.float64)

  def forward(self,X):
    z = torch.matmul(X,self.weights)+self.bias
    y_pred = torch.sigmoid(z)
    return y_pred # Return probabilities after sigmoid activation

  def loss(self,y_true,y_pred):
    epsilon = 1e-17
    y_pred = torch.clip(y_pred,epsilon,1 - epsilon)

    loss = -torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
    return loss

In [57]:
# Now instantiating the hyperparameters
learning_rate = 0.1
epochs = 25

In [58]:
# create model
model = SimpleNN(X_train,y_train)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train) # y_pred will now be probabilities

  # loss calculate
  loss = model.loss(y_train, y_pred) # Pass y_train as y_true, y_pred as predicted probabilities

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 1.3108669562281206
Epoch: 2, Loss: 1.2978072026421013
Epoch: 3, Loss: 1.284932393732439
Epoch: 4, Loss: 1.2722390183179682
Epoch: 5, Loss: 1.2597237286141227
Epoch: 6, Loss: 1.247383343048251
Epoch: 7, Loss: 1.235214847696617
Epoch: 8, Loss: 1.2232153965071655
Epoch: 9, Loss: 1.2113823104634331
Epoch: 10, Loss: 1.1997130758345584
Epoch: 11, Loss: 1.1882053416446017
Epoch: 12, Loss: 1.1768569164818246
Epoch: 13, Loss: 1.165665764755909
Epoch: 14, Loss: 1.1546300024983753
Epoch: 15, Loss: 1.1437478927891769
Epoch: 16, Loss: 1.1330178408808191
Epoch: 17, Loss: 1.1224383890804837
Epoch: 18, Loss: 1.112008211440579
Epoch: 19, Loss: 1.1017261082991805
Epoch: 20, Loss: 1.0915910007036855
Epoch: 21, Loss: 1.0816019247439461
Epoch: 22, Loss: 1.071758025814967
Epoch: 23, Loss: 1.0620585528240136
Epoch: 24, Loss: 1.0525028523525544
Epoch: 25, Loss: 1.0430903627798482


In [59]:
model.weights

tensor([[ 0.1276],
        [ 0.5962],
        [-0.3824],
        [ 0.1590],
        [ 0.0034],
        [-0.5018],
        [ 0.4150],
        [ 0.7892],
        [-1.1688],
        [-0.7353]], dtype=torch.float64, requires_grad=True)

In [60]:
model.bias

tensor([0.0142], dtype=torch.float64, requires_grad=True)

In [61]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.47999998927116394
